# E19 · Class-prior adjustment of the best ensemble (E15d)

Every model is trained on EuroSAT's class proportions (each class 7–11%) and implicitly assumes the Kaggle images have
the same mix. They might not: in E16, 1,020 Kaggle images (24%) were predicted SeaLake with all 8 views agreeing,
against 11% SeaLake in EuroSAT. If the true Kaggle mix differs, the decision boundaries are shifted and some
borderline images go to the wrong class.

E19 adjusts the **E15d** probabilities (72.95%, best so far) without any training:

* **E19a · EM prior estimation** (Saerens, Latinne & Decaestecker, 2002). Estimate the Kaggle class mix from the
  model's own probabilities, then re-weight every prediction by *(Kaggle share / training share)* of its class;
  repeat until the estimate stops changing. Label-free and standard for train/test prior shift.
* **E19b · Match the EuroSAT mix.** The opposite assumption: the Kaggle images were sampled like EuroSAT, so the
  predicted class counts should match EuroSAT's. Class weights are tuned until they do.

E19a and E19b move in opposite directions where the predicted mix differs from EuroSAT, so their two scores tell us
which assumption is closer to the truth. A half-strength EM version (E19c) is computed for reference.

**No training**: the four E15d members are loaded and predict as in E15 (a few minutes).

## 1 · Setup and E15d probabilities

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
sys.path.insert(0, str(ROOT))
from src.data import CLASSES, PROJECT_ROOT, TEST_BANDS
from src.experiment import class_mix, find_run, kaggle_score, load_experiment_data, plot_class_mix
from src.plot_style import apply_style
from src.training import load_run, new_run_dir, predict_views, write_submission
from src.utils import describe_device, get_device

apply_style()

# ---- Experiment settings ---------------------------------------------------------------
NOTEBOOK = "E19_prior_adjustment"
E15D_MEMBERS = {   # exactly the E15d ensemble (72.95%)
    "E09a": "pl_r3_view_agree_contrast_aug_hist_split_effb0",
    "E12a+AdaBN": "adabn_balanced_pl_r4_view_agree_contrast_aug_hist_split_effb0",
    "E13a+AdaBN": "adabn_balanced_indices_pl_r3_view_agree_contrast_aug_hist_split_effb0",
    "E17a": "l2a_pl_r3_view_agree_contrast_aug_effb0",
}
E15D_RUN = "ensemble_e15d_e09a_e12a_e13a_e17a"
SAVE = {"E19a": None, "E19b": None}   # saved folders; None = create a new folder
# -----------------------------------------------------------------------------------------

device = get_device()
print("Device:", describe_device(device))
data = load_experiment_data(list(TEST_BANDS), ["hist_split"], device)

views = []
for name, run in E15D_MEMBERS.items():
    model, normalizer, config = load_run(find_run(run), device)
    idx = torch.tensor([TEST_BANDS.index(b) for b in config["bands"]], device=device)
    views.append(predict_views(model, data.kg_images[:, idx], normalizer))
    del model
    if device.type == "mps":
        torch.mps.empty_cache()
prob = np.mean(views, axis=0).mean(0)          # (N, 10): E15d probabilities

# check: same answers as the submitted E15d file
e15d_dir = find_run(E15D_RUN)
submitted = pd.read_csv(e15d_dir / "submission.csv")["label"].values
same = (np.array(CLASSES)[prob.argmax(1)] == submitted).mean()
print(f"Reproduced E15d: {same:.2%} identical to its submission (Kaggle {kaggle_score(e15d_dir)}%)")

## 2 · Prior adjustment

Training prior = class shares of the EuroSAT training split. (The members also saw ~3,000 pseudo-labelled Kaggle
images; they shift the prior by well under a percentage point and are ignored.)

In [ ]:
def em_priors(prob, train_prior, iters=500, tol=1e-8):
    """Saerens et al. (2002): estimate the test priors; returns (test prior, adjusted probabilities)."""
    prior = train_prior.copy()
    for _ in range(iters):
        p = prob * (prior / train_prior)
        p /= p.sum(1, keepdims=True)
        new = p.mean(0)
        done = np.abs(new - prior).max() < tol
        prior = new
        if done:
            break
    return prior, p


def reweight(prob, weights):
    p = prob * weights
    return p / p.sum(1, keepdims=True)


def match_counts(prob, target_share, iters=500, step=0.5):
    """Class weights that make the arg-max class counts match target_share."""
    logw = np.zeros(prob.shape[1])
    logp = np.log(prob + 1e-12)
    for _ in range(iters):
        share = np.bincount((logp + logw).argmax(1), minlength=prob.shape[1]) / len(prob)
        logw += step * (np.log(target_share + 1e-6) - np.log(share + 1e-6))
        logw -= logw.mean()
    return np.exp(logw)


train_prior = np.bincount(data.y_tr, minlength=10) / len(data.y_tr)
em_prior, prob_a = em_priors(prob, train_prior)
prob_b = reweight(prob, match_counts(prob, train_prior))
prob_c = reweight(prob, np.sqrt(em_prior / train_prior))        # half-strength EM, for reference
adjusted = {"E19a": prob_a, "E19b": prob_b, "E19c (half EM)": prob_c}

mix = pd.DataFrame({"EuroSAT train": train_prior * 100, "E15d predicted": class_mix(prob).values,
                    "EM estimate of Kaggle": em_prior * 100,
                    **{k: class_mix(p).values for k, p in adjusted.items()}}, index=CLASSES)
mix.round(1)

In [ ]:
pred = prob.argmax(1)
summary = pd.DataFrame({k: {"answers changed vs E15d": int((p.argmax(1) != pred).sum()),
                            "share changed (%)": (p.argmax(1) != pred).mean() * 100,
                            "mean confidence (%)": p.max(1).mean() * 100} for k, p in adjusted.items()}).T
summary.round(2)

In [ ]:
# Changed answers: E15d's class (rows) -> adjusted class (columns)
names = np.array(CLASSES)
for k in ["E19a", "E19b"]:
    new = adjusted[k].argmax(1)
    changed = new != pred
    print(f"{k}: {changed.sum()} answers changed")
    display(pd.crosstab(names[pred[changed]], names[new[changed]]).rename_axis(index="E15d", columns=k))

In [ ]:
plot_class_mix(mix[["EuroSAT train", "E15d predicted", "EM estimate of Kaggle", "E19a", "E19b"]])

## 3 · Submission files

In [ ]:
for variant, description in [("E19a", "E15d with EM prior adjustment (Saerens et al. 2002)"),
                             ("E19b", "E15d with class weights matching the EuroSAT class mix")]:
    if SAVE[variant]:
        out = PROJECT_ROOT / SAVE[variant]
    else:
        out = new_run_dir(f"prior_{variant.lower()}_ensemble_e15d")
        (out / "config.json").write_text(json.dumps({
            "run_name": out.name, "notebook": NOTEBOOK, "variant": variant, "description": description,
            "base": str(e15d_dir.relative_to(PROJECT_ROOT)), "members": list(E15D_MEMBERS.values()),
            "train_prior": train_prior.round(5).tolist(), "em_prior": em_prior.round(5).tolist()}, indent=2))
        (out / "metrics.json").write_text(json.dumps({
            "kaggle_predicted_mix_pct": class_mix(adjusted[variant]).round(2).to_dict(),
            "answers_changed_vs_e15d": int((adjusted[variant].argmax(1) != pred).sum()),
            "kaggle_public_score": None}, indent=2))
    write_submission(adjusted[variant], data.test_ids, out / "submission.csv")
    print(f"{variant}: {out.relative_to(PROJECT_ROOT)}/submission.csv")

## 4 · Conclusion

*To fill in after the Kaggle submission.*

| Variant | Assumption about the Kaggle class mix | Kaggle public |
|---|---|---|
| E15d | same as EuroSAT, no adjustment | **72.95%** |
| E19a | estimated from the predictions (EM) | |
| E19b | exactly EuroSAT's mix | |

If E19a wins: the Kaggle mix differs from EuroSAT; try the same adjustment on other strong models.
If E19b wins: the models over-predict some classes; matching the training mix is the better default.
If both lose: the class mix is not the problem.